# SIRA

Welcome! This notebook is my solution to the evaluation test for the HumanAI GSoC project **"Learning the Susceptible-Infected-Removed (SIR) Model"**, also referred to as **SIRA**.  

The goal is to:
- Generate realistic synthetic epidemic data using a **stochastic SIR model**.
- Fit a **neural network** to the simulated trajectories of S(t), I(t), and R(t).
- Use **symbolic regression** to recover interpretable equations that approximate the underlying epidemic dynamics.

Along the way, I comment on design choices, limitations, and ideas for improving the pipeline—so this is not just code, but also a readable narrative of my thought process.

## Notebook Structure

1. **Introduction**  
   What this notebook aims to do and how it fits into the broader SIRA project.

2. **SIR Simulator**  
   A stochastic SIR simulator to generate synthetic epidemic data, with visualizations to build intuition.

3. **Machine Learning Model**  
   A neural network that learns to approximate the trajectories S(t), I(t), and R(t) from the simulated data.

4. **Symbolic Regression**  
   Applying PySR to extract compact, human-readable equations capturing the epidemic dynamics.

5. **Conclusion & Reflection**  
   A quick summary of what worked, what could be improved, and how I would extend this in a full project.

6. **Appendix / Bonus Experiments**  
   Extra steps: multiple stochastic runs, derivative-based symbolic regression, and evaluation metrics.

## 🧪 SIR Simulator: Generate and Visualize Synthetic Epidemic Data

We start with a simple **stochastic SIR simulator**. Instead of using the deterministic ODE version, we simulate random infection and recovery events using binomial draws. This better mimics noisy real-world dynamics while still being grounded in the classical SIR structure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# For reproducibility of the stochastic simulations
np.random.seed(42)

def simulate_sir(S0, I0, R0, beta, gamma, max_time, dt=1.0):
    """
    Simulate a stochastic SIR process using binomial infection and recovery events.

    Parameters
    ----------
    S0, I0, R0 : int
        Initial susceptible, infectious, and recovered counts.
    beta : float
        Infection rate parameter.
    gamma : float
        Recovery rate parameter.
    max_time : float
        Maximum simulation time.
    dt : float, optional
        Time step size (default: 1.0).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['time', 'S', 'I', 'R'].
    """
    times = [0]
    S_vals = [S0]
    I_vals = [I0]
    R_vals = [R0]

    S, I, R = S0, I0, R0
    t = 0

    N = S0 + I0 + R0

    while t < max_time and I > 0:
        # Effective infection and recovery probabilities over time step dt
        inf_prob = 1 - np.exp(-beta * I / N * dt)
        rec_prob = 1 - np.exp(-gamma * dt)

        new_infections = np.random.binomial(S, inf_prob)
        new_recoveries = np.random.binomial(I, rec_prob)

        S = max(S - new_infections, 0)
        I = max(I + new_infections - new_recoveries, 0)
        R = min(R + new_recoveries, N)

        t += dt
        times.append(t)
        S_vals.append(S)
        I_vals.append(I)
        R_vals.append(R)

    return pd.DataFrame({
        "time": times,
        "S": S_vals,
        "I": I_vals,
        "R": R_vals
    })

# Run a baseline simulation
S0, I0, R0 = 990, 10, 0
beta, gamma = 0.3, 0.1
max_time = 100

df = simulate_sir(S0, I0, R0, beta, gamma, max_time)
df.head()

: 

In [ ]:
# Visualize the stochastic SIR trajectory
plt.figure(figsize=(10, 6))
plt.plot(df['time'], df['S'], label='Susceptible (S)')
plt.plot(df['time'], df['I'], label='Infectious (I)')
plt.plot(df['time'], df['R'], label='Recovered (R)')
plt.xlabel('Time')
plt.ylabel('Population')
plt.title('Stochastic SIR Simulation')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 🤖 Machine Learning Model: Neural Network Fit for SIR Curves

Next, I treat **time** as the input and learn a mapping 
\( t \mapsto (S(t), I(t), R(t)) \) using a small feed-forward neural network.

This is intentionally simple: the goal is not to build the most powerful forecaster, but to show that a generic neural network can approximate the simulated SIR trajectories.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler

# Normalize time and population values for stable NN training
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['time', 'S', 'I', 'R']])
df_scaled = pd.DataFrame(scaled_data, columns=['time', 'S', 'I', 'R'])

# Prepare training data: input t → output (S, I, R)
X = torch.tensor(df_scaled['time'].values.reshape(-1, 1), dtype=torch.float32)
y = torch.tensor(df_scaled[['S', 'I', 'R']].values, dtype=torch.float32)

In [ ]:
class SIRNet(nn.Module):
    """A small fully connected network to learn S(t), I(t), R(t) from time."""
    def __init__(self):
        super(SIRNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.net(x)

model = SIRNet()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# Train the model
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

In [ ]:
# Predict and inverse transform the results back to the original scale
model.eval()
with torch.no_grad():
    predictions = model(X).numpy()

# Inverse transform: we reconstruct a DataFrame with time + predictions
pred_df = pd.DataFrame(predictions, columns=['S', 'I', 'R'])
pred_df['time'] = df_scaled['time']
recovered = scaler.inverse_transform(pred_df[['time', 'S', 'I', 'R']])
pred_df = pd.DataFrame(recovered, columns=['time', 'S', 'I', 'R'])

# Plot predictions vs original simulation
plt.figure(figsize=(10, 6))
plt.plot(df['time'], df['S'], label='True S', alpha=0.6)
plt.plot(df['time'], df['I'], label='True I', alpha=0.6)
plt.plot(df['time'], df['R'], label='True R', alpha=0.6)

plt.plot(pred_df['time'], pred_df['S'], '--', label='Predicted S')
plt.plot(pred_df['time'], pred_df['I'], '--', label='Predicted I')
plt.plot(pred_df['time'], pred_df['R'], '--', label='Predicted R')

plt.xlabel('Time')
plt.ylabel('Population')
plt.title('Neural Network Fit to SIR Simulation')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## ⚙️ Auto-Differentiation: Derivatives from the Neural Network

In addition to training the neural network with backpropagation, I explicitly use **PyTorch autograd** here to compute

the time derivatives of the learned trajectories:

\\( dS/dt, dI/dt, dR/dt \\) as implied by the network.



This makes the use of **auto-differentiation** explicit in the notebook and provides derivative estimates that could

be fed into symbolic regression pipelines as well.

In [ ]:
# Use PyTorch autograd to compute dS/dt, dI/dt, dR/dt from the trained network

model.eval()



# Clone the (normalized) time input and enable gradients

t_ad = X.clone().detach().requires_grad_(True)

pred_ad = model(t_ad)  # shape: (T, 3) -> [S(t), I(t), R(t)] in normalized space



# Compute gradients of each output component with respect to time

dS_dt_ad = torch.autograd.grad(pred_ad[:, 0].sum(), t_ad, retain_graph=True)[0].detach().numpy().flatten()

dI_dt_ad = torch.autograd.grad(pred_ad[:, 1].sum(), t_ad, retain_graph=True)[0].detach().numpy().flatten()

dR_dt_ad = torch.autograd.grad(pred_ad[:, 2].sum(), t_ad)[0].detach().numpy().flatten()



# Pack into a DataFrame for convenience / potential downstream symbolic regression

autodiff_df = pd.DataFrame({

    't_normalized': t_ad.detach().numpy().flatten(),

    'S_norm': pred_ad[:, 0].detach().numpy().flatten(),

    'I_norm': pred_ad[:, 1].detach().numpy().flatten(),

    'R_norm': pred_ad[:, 2].detach().numpy().flatten(),

    'dS_dt_norm': dS_dt_ad,

    'dI_dt_norm': dI_dt_ad,

    'dR_dt_norm': dR_dt_ad,

})



autodiff_df.head()

## 🧮 Symbolic Regression: Extract Equations from Predictions using PySR

Now that we have a smooth neural approximation of the SIR trajectories, we can ask a more ambitious question:

> Can we rediscover compact, human-readable equations that describe these curves?

For this, I use **PySR**, a symbolic regression library that searches over expressions built from basic operators and functions.

In [ ]:
from pysr import PySRRegressor

# Prepare data for symbolic regression: time → predicted values
X_sym = pred_df[['time']].values
y_sym = pred_df[['S', 'I', 'R']].values

# Fit symbolic regression model separately for S(t), I(t), R(t)
equations = []
for i, label in enumerate(['S', 'I', 'R']):
    model_sym = PySRRegressor(
        niterations=40,
        binary_operators=['+', '-', '*', '/'],
        unary_operators=['sin', 'cos', 'exp'],
        model_selection='best',
        loss='loss(x, y) = (x - y)^2',
        maxsize=20,
    )
    print(f"Fitting symbolic model for {label}(t)...")
    model_sym.fit(X_sym, y_sym[:, i])
    equations.append((label, model_sym))

# Display discovered equations
for label, model_sym in equations:
    print(f"Symbolic expression for {label}(t):")
    print(model_sym)

## ✅ Conclusion & Reflection

In this evaluation notebook, I walked through a compact but complete pipeline:

- **Simulated** a stochastic SIR process with configurable parameters.
- **Trained** a neural network to approximate S(t), I(t), R(t) over time.
- **Applied symbolic regression** (PySR) to extract interpretable formulas that approximate the learned trajectories.

Together, these steps show how **data-driven** and **symbolic** methods can complement each other: simulations give us data, neural nets give us flexible function approximators, and symbolic regression pulls us back toward **human-understandable mathematics**.

### Possible Future Improvements

- Improve robustness by simulating **multiple epidemic runs** and averaging trajectories.
- Target the **differential equations directly** by modeling dS/dt, dI/dt, dR/dt rather than S, I, R alone.
- Quantitatively compare discovered symbolic models against the **known analytical SIR ODEs**.

In [ ]:
# Stage 1 (Appendix): Multiple stochastic simulations with averaging

def simulate_sir(S0, I0, R0, beta, gamma, max_time, dt=1.0):
    times = [0]
    S_vals = [S0]
    I_vals = [I0]
    R_vals = [R0]
    S, I, R = S0, I0, R0
    t = 0
    N = S0 + I0 + R0
    while t < max_time and I > 0:
        inf_prob = 1 - np.exp(-beta * I / N * dt)
        rec_prob = 1 - np.exp(-gamma * dt)
        new_infections = np.random.binomial(S, inf_prob)
        new_recoveries = np.random.binomial(I, rec_prob)
        S = max(S - new_infections, 0)
        I = max(I + new_infections - new_recoveries, 0)
        R = min(R + new_recoveries, N)
        t += dt
        times.append(t)
        S_vals.append(S)
        I_vals.append(I)
        R_vals.append(R)
    return pd.DataFrame({"time": times, "S": S_vals, "I": I_vals, "R": R_vals})

def simulate_many_runs(num_runs=100, S0=990, I0=10, R0=0, beta=0.3, gamma=0.1, max_time=100, dt=1.0):
    all_S, all_I, all_R = [], [], []
    time_reference = np.arange(0, max_time + dt, dt)
    for _ in range(num_runs):
        df_run = simulate_sir(S0, I0, R0, beta, gamma, max_time, dt)
        padded = pd.DataFrame({'time': time_reference})
        merged = pd.merge(padded, df_run, on='time', how='left').interpolate(limit_direction='forward').fillna(method='bfill')
        all_S.append(merged['S'].values)
        all_I.append(merged['I'].values)
        all_R.append(merged['R'].values)
    S_matrix = np.stack(all_S)
    I_matrix = np.stack(all_I)
    R_matrix = np.stack(all_R)
    return pd.DataFrame({
        'time': time_reference,
        'S_mean': np.nanmean(S_matrix, axis=0),
        'S_std': np.nanstd(S_matrix, axis=0),
        'I_mean': np.nanmean(I_matrix, axis=0),
        'I_std': np.nanstd(I_matrix, axis=0),
        'R_mean': np.nanmean(R_matrix, axis=0),
        'R_std': np.nanstd(R_matrix, axis=0),
    })

summary = simulate_many_runs()

## 📈 Neural Network on Mean S, I, R (Multiple Runs)

In the appendix above, I averaged over many stochastic simulations to obtain mean trajectories

\\( S_\\text{mean}(t), I_\\text{mean}(t), R_\\text{mean}(t) \\).



To align with the requirement of predicting **mean counts**, I now train the same neural architecture

directly on these averaged curves. This shows that the model can learn not only a single noisy

realization, but also the smoother, population-level behaviour.

In [ ]:
# Train a neural network on the mean trajectories S_mean, I_mean, R_mean

from sklearn.preprocessing import MinMaxScaler



# Scale time and mean counts for stable training

scaler_mean = MinMaxScaler()

scaled_summary = scaler_mean.fit_transform(summary[['time', 'S_mean', 'I_mean', 'R_mean']])



df_mean_scaled = pd.DataFrame(

    scaled_summary, columns=['time', 'S_mean', 'I_mean', 'R_mean']

)



X_mean = torch.tensor(df_mean_scaled['time'].values.reshape(-1, 1), dtype=torch.float32)

y_mean = torch.tensor(

    df_mean_scaled[['S_mean', 'I_mean', 'R_mean']].values, dtype=torch.float32

)



# Reuse the SIRNet architecture defined above

model_mean = SIRNet()

criterion_mean = nn.MSELoss()

optimizer_mean = optim.Adam(model_mean.parameters(), lr=0.01)



epochs_mean = 500

for epoch in range(epochs_mean):

    model_mean.train()

    optimizer_mean.zero_grad()

    outputs_mean = model_mean(X_mean)

    loss_mean = criterion_mean(outputs_mean, y_mean)

    loss_mean.backward()

    optimizer_mean.step()

    if epoch % 100 == 0:

        print(f"[Mean NN] Epoch {epoch}, Loss: {loss_mean.item():.6f}")

In [ ]:
# Stage 2: Compute time derivatives and fit symbolic regression models on dynamics
dt = 1.0
S = summary['S_mean'].values
I = summary['I_mean'].values
R = summary['R_mean'].values
time = summary['time'].values
dS_dt = np.gradient(S, dt)
dI_dt = np.gradient(I, dt)
dR_dt = np.gradient(R, dt)
X_dyn = np.stack([S, I, R], axis=1)

from pysr import PySRRegressor

model_dS = PySRRegressor(niterations=40, binary_operators=['+', '-', '*'], unary_operators=[], model_selection='best')
model_dS.fit(X_dyn, dS_dt)

model_dI = PySRRegressor(niterations=40, binary_operators=['+', '-', '*'], unary_operators=[], model_selection='best')
model_dI.fit(X_dyn, dI_dt)

model_dR = PySRRegressor(niterations=40, binary_operators=['+', '-', '*'], unary_operators=[], model_selection='best')
model_dR.fit(X_dyn, dR_dt)

display(model_dS.equations_)
display(model_dI.equations_)
display(model_dR.equations_)

In [ ]:
# Stage 3: Placeholder for an LSTM/GRU comparison model
print("This is where an LSTM/GRU model could be added to compare forecasting results.")

In [ ]:
# Stage 4: Evaluation metrics for the learned dynamics models
from sklearn.metrics import mean_squared_error, r2_score

for name, y_true, y_pred in zip(
    ['S', 'I', 'R'],
    [dS_dt, dI_dt, dR_dt],
    [model_dS.predict(X_dyn), model_dI.predict(X_dyn), model_dR.predict(X_dyn)],
):
    print(f"{name} RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))
    print(f"{name} R²:", r2_score(y_true, y_pred))

In [ ]:
# Stage 5: Display the classical SIR model equations for reference
from IPython.display import Markdown
Markdown(r'''
### SIR Model Equations

The standard SIR model is a system of differential equations:

$$
\begin{aligned}
\frac{dS}{dt} &= -\beta S I \\n
\frac{dI}{dt} &= \beta S I - \gamma I \\n
\frac{dR}{dt} &= \gamma I
\end{aligned}
$$

Where:

- \( S \): Susceptible population  \n
- \( I \): Infected population  \n
- \( R \): Recovered population  \n
- \( \beta \): Transmission rate  \n
- \( \gamma \): Recovery rate
''')

In [ ]:
# (Optional) scratch cell for further experiments
# Feel free to use this cell to tweak parameters,
# try different neural architectures, or explore new ideas.
